# UNDERTONE - analysis

Reads every `results/<model>.jsonl` the thirteen sweeps produced and builds the
paper tables.

Three rules the retired analysis broke and this one does not:

1. **Truncated cells never enter an accuracy table.** "This model cannot ingest
   the audio" and "this model heard it and got it wrong" are different findings.
   Truncation gets its own table, reported as coverage.
2. **Unverified items are excluded and counted.** A proposal nobody has listened
   to is not evidence.
3. **No composite score.** The paper plan says so, and the retired suite's
   composites hid which term was zero - every reported `0.000` there meant
   "nothing parsed", not "no error".

## The findings these tables test

| | claim | where to look |
|---|---|---|
| F1 | type dominates duration | spread across Category at fixed band vs spread across band at fixed Category |
| F2 | the salience prior is the mechanism | `salience_trap` on P3 vs chance (0.25) |
| F3' | the prior is English-shaped | `salience_trap` on P3, hi and bn vs en |
| F4 | perception is intact, retrieval is not | `RetrievalCost` large while `acc_L1` is high |
| F5 | the mechanism is specific | P1-P4 share the trap signature; C1 does not |

F3' is weaker than the paper plan's F3, which needed a tone language. en/hi/bn
has none: English marks prominence with lexically free stress-accent, while
Hindi and Bengali have weak or fixed word stress plus grammaticalised focus
(`hi`/`to`/`bhi`; `-i`/`-o`; word order). Both the substitution and the fact that
Hindi and Bengali are the same family belong in Limitations.


In [ ]:
# Pinned for this model. If `load()` fails, this cell is the first thing to change.
%pip install -q "pandas>=2.1.0"
%pip install -q "matplotlib>=3.8.0"
print("--- resolved versions (freeze these before the paper run) ---")
import importlib.metadata as md
for pkg in ["transformers", "accelerate", "torch", "librosa"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} not installed")

In [ ]:
import os, random, sys, json
import numpy as np, torch

SEED = 20260904
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Weights go to /kaggle/temp: scratch, and it does NOT count against the 20 GB
# /kaggle/working output cap. A 16-18 GB checkpoint in /kaggle/working would
# fail the commit at the end of the session.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm{p.major}{p.minor}")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    print("\nsm < 80: no bf16 compute and no flash-attention-2. "
          "Every adapter loads in fp16 for this reason.")

In [ ]:
REPO_URL = "https://github.com/DeepanIsCool/longaudiobench.git"
REPO_REF = "undertone"   # pin to a commit sha before the paper run

import subprocess, shutil, os, sys
if os.path.exists("/kaggle/working/longaudiobench"):
    shutil.rmtree("/kaggle/working/longaudiobench")
for attempt in range(3):
    rc = subprocess.call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                          REPO_URL, "/kaggle/working/longaudiobench"])
    if rc == 0:
        break
else:
    raise RuntimeError("could not clone the benchmark repo")

sys.path.insert(0, "/kaggle/working/longaudiobench")
import importlib; importlib.invalidate_caches()

from undertone import ItemPack, adapters, env, runner, scoring
print("adapters registered:", len(adapters.list_adapters()))

if env.export_hf_token():
    print("HF token resolved")
elif globals().get("GATED"):
    raise RuntimeError(
        "this model is gated and no token was found. Add a Kaggle secret named "
        "HF_TOKEN, or write the token to .hf_token at the repo root.")

hw = env.resolve_hardware()
print(f"hardware: {hw.detail}  dtype={hw.dtype}  signature={hw.signature}")
print(f"versions: {env.versions()}")
# Every result row is stamped with this signature. The analysis refuses to put
# two signatures in one table -- a benchmark whose rows came from different
# backends compares machines, not models.

In [ ]:
import glob
from undertone import analysis, runner
from undertone.adapters.base import get_adapter

rows = []
for path in sorted(glob.glob("/kaggle/input/*/results/*.jsonl")
                   + glob.glob("/kaggle/working/results/*.jsonl")):
    rows.extend(runner.load_rows(path))
print(f"{len(rows)} rows from {len({r['model_key'] for r in rows})} models")

problems = analysis.sanity_checks(rows)
for p in problems:
    print("SANITY:", p)
if not problems:
    print("sanity checks clean")

In [ ]:
import pandas as pd

limits = {k: get_adapter(k).max_audio_s for k in {r["model_key"] for r in rows}}

t4 = pd.DataFrame(analysis.table4_truncation(rows, limits))
print("Table 4 - per-model input limits and truncation coverage")
print(t4.to_string(index=False))

t1 = pd.DataFrame(analysis.table1_main(rows))
print("\nTable 1 - main results at L3 (salience_trap is the headline)")
print(t1.to_string(index=False))

t2 = pd.DataFrame(analysis.table2_ladder(rows))
print("\nTable 2 - ladder decomposition")
print(t2.to_string(index=False))

print("\nNull items")
print(pd.DataFrame(analysis.table1_nulls(rows)).to_string(index=False))

recovery_path = "/kaggle/input/undertone-item-pack/needle_recovery.json"
if os.path.exists(recovery_path):
    print("\nAudio necessity - share of items whose answer ASR never wrote down")
    print(pd.DataFrame(json.load(open(recovery_path))).to_string(index=False))

print("\nScorer gap: letter logits vs free generation")
print(pd.DataFrame(analysis.scorer_gap(rows)).to_string(index=False))

from undertone.analysis import figures
os.makedirs("/kaggle/working/analysis", exist_ok=True)
made = figures.all_figures(rows, "/kaggle/working/analysis")
print("figures:", [p.name for p in made])

for name, frame in [("table1_main", t1), ("table2_ladder", t2),
                    ("table4_truncation", t4),
                    ("table_language", pd.DataFrame(analysis.table_language(rows)))]:
    frame.to_csv(f"/kaggle/working/analysis/{name}.csv", index=False)
    with open(f"/kaggle/working/analysis/{name}.tex", "w") as fh:
        fh.write(frame.to_latex(index=False))
print("\nwrote /kaggle/working/analysis/")

In [ ]:
from undertone import scoring
# F2: does the salience trap exist at all? This is the paper plan's kill point.
p3 = [r for r in analysis.usable(rows) if r["category"] == "P3" and r["condition"] == "L3"]
for model in sorted({r["model_key"] for r in p3}):
    cell = [r for r in p3 if r["model_key"] == model]
    rate = scoring.salience_trap(cell)
    lo, hi = scoring.cluster_bootstrap_ci(cell, scoring.salience_trap)
    verdict = "above chance" if lo > 0.25 else "NOT above chance"
    print(f"{model:26s} P3 salience trap {rate:.3f} [{lo:.3f}, {hi:.3f}]  {verdict}")

# F5: the mechanism has to be specific. P1-P4 share the signature; C1 must not.
print()
for category in ["P1", "P2", "P3", "P4", "C1"]:
    cell = [r for r in analysis.usable(rows)
            if r["category"] == category and r["condition"] == "L3"]
    if cell:
        print(f"{category}  salience {scoring.salience_trap(cell):.3f}  "
              f"acc {scoring.accuracy(cell):.3f}  n={len(cell)}")

# F3': is the prior English-shaped? Weaker than the plan's F3 - no tone language.
print()
lang_table = analysis.table_language(rows)
for lang in ("en", "hi", "bn"):
    cells = [r for r in lang_table if r["lang"] == lang and r["category"] == "P3"]
    if cells:
        mean = sum(c["salience_trap"] for c in cells) / len(cells)
        print(f"{lang}  P3 salience trap {mean:.3f}  ({len(cells)} model cells)")